In [16]:
import pandas as pd
import yaml
from pathlib import Path

from src.data import load_dataset, split_dataset
from src.utils import get_git_sha, sha256_file

In [ ]:
cfg = yaml.safe_load(Path("/home/trevi/projects/mlops-examples/configs/dev.yaml").read_text())
1
data_path = Path(cfg["data"]["path"])
out_dir = Path(cfg["artifacts"]["out_dir"])

# ── Data ────────────────────────────────────────────────────────────
df = load_dataset(data_path)
data_hash = sha256_file(data_path)
git_sha = get_git_sha()

# # ── Featurize ────────────────────────────────────────────────────────────
features_df = df.drop("target", axis=1)
target_df = df["target"]

timestamps = pd.date_range(end = pd.Timestamp.now(),
                           periods = len(df), freq="D").to_frame(name = 'event_timestamp', index = False)

features_df = pd.concat(objs = [features_df, timestamps], axis = 1)
target_df = pd.concat(objs = [target_df, timestamps], axis = 1)

dataLen = len(df)
idsList = list(range(dataLen))

patient_ids = pd.DataFrame(data = idsList, columns = ['patient_id'])

features_df = pd.concat(objs = [features_df, patient_ids], axis = 1)
target_df = pd.concat(objs = [target_df, patient_ids], axis = 1)

In [21]:
features_df.to_parquet(path='feature_repo/data/features_df.parquet')
target_df.to_parquet(path='feature_repo/data/target_df.parquet')

In [ ]:
from feast import FeatureStore
from feast.infra.offline_stores.file_source import SavedDatasetFileStorage

store = FeatureStore(repo_path="feature_repo")
entity_df = pd.read_parquet('feature_repo/data/target_df.parquet')

training_data = store.get_historical_features(
    entity_df = entity_df,
    features = [
        'features_df_feature_view:mean radius',
        'features_df_feature_view:mean texture',
        'features_df_feature_view:mean perimeter'
    ]
)

dataset = store.create_saved_dataset(
    from_ = training_data,
    name = 'cancer_dataset',
    storage = SavedDatasetFileStorage('data/cancer_dataset.parquet')
)

In [ ]:
df = store.get_saved_dataset(name='cancer_dataset').to_df()

In [ ]:
# X = df.drop(columns=['target', 'event_timestamp', 'patient_id'])
# y = df['target']

# X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)